# 03 — Dashboard
**Indian Equities Analytics & Direction Predictor**

The full interactive dashboard is a **Streamlit app** at the repo root: `app.py`.

Run it with:
```bash
streamlit run app.py
```

It lets you pick a stock and see: candlestick price chart with SMA/EMA overlays,
RSI/MACD/Bollinger-width indicator panels, tomorrow's predicted direction with
probability, and historical model accuracy across all 5 stocks.

This notebook gives a **static, notebook-native preview** of the same
components using Plotly, useful for grading/review without needing to launch
a server.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import joblib
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data import fetch_stock, STOCKS
from src.features import build_features, FEATURE_COLUMNS

TICKER = 'RELIANCE.NS'  # change to preview a different stock
raw = fetch_stock(TICKER)
feats = build_features(raw, ticker=TICKER)
print(raw.shape, feats.shape, 'source:', raw.attrs.get('source', 'cached'))


(1306, 5) (1257, 12) source: cached


## Price chart with moving averages (last 250 sessions)

In [2]:
plot_df = raw.tail(250).copy()
plot_df['SMA_10'] = raw['Close'].rolling(10).mean().tail(250)
plot_df['SMA_50'] = raw['Close'].rolling(50).mean().tail(250)
plot_df['EMA_20'] = raw['Close'].ewm(span=20, adjust=False).mean().tail(250)

fig = go.Figure()
fig.add_trace(go.Candlestick(x=plot_df.index, open=plot_df['Open'], high=plot_df['High'],
                              low=plot_df['Low'], close=plot_df['Close'], name='Price'))
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['SMA_10'], name='SMA 10', line=dict(width=1)))
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['SMA_50'], name='SMA 50', line=dict(width=1)))
fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['EMA_20'], name='EMA 20', line=dict(width=1, dash='dot')))
fig.update_layout(title=f'{TICKER} Price + Moving Averages', height=450, xaxis_rangeslider_visible=False)
fig.show()


## Technical indicator panels

In [3]:
ind_df = feats.tail(250)
ind_fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[0.34, 0.33, 0.33],
                         subplot_titles=('RSI (14)', 'MACD Histogram', 'Bollinger Band Width'))
ind_fig.add_trace(go.Scatter(x=ind_df.index, y=ind_df['rsi_14'], name='RSI 14'), row=1, col=1)
ind_fig.add_hline(y=70, line_dash='dot', line_color='red', row=1, col=1)
ind_fig.add_hline(y=30, line_dash='dot', line_color='green', row=1, col=1)
ind_fig.add_trace(go.Bar(x=ind_df.index, y=ind_df['macd_hist'], name='MACD hist'), row=2, col=1)
ind_fig.add_trace(go.Scatter(x=ind_df.index, y=ind_df['bb_width'], name='BB width'), row=3, col=1)
ind_fig.update_layout(height=550, showlegend=False, title=f'{TICKER} Indicators')
ind_fig.show()


## Next-day prediction from the saved best model

In [4]:
bundle = joblib.load('../models/best_model.joblib')
model, model_name = bundle['model'], bundle['name']

latest_row = feats.iloc[[-1]][FEATURE_COLUMNS]
proba_up = model.predict_proba(latest_row)[0][1]
direction = 'UP' if proba_up >= 0.5 else 'DOWN'

print(f'Model        : {model_name}')
print(f'As of         : {feats.index[-1].date()}')
print(f'Predicted     : {direction}')
print(f'P(up)         : {proba_up:.1%}')


Model        : gradient_boosting
As of         : 2024-07-01
Predicted     : UP
P(up)         : 54.0%


## Notes

- All charts above are generated with the exact same `src/data.py` and
  `src/features.py` functions used by `app.py`, so behavior is identical
  between the notebook preview and the live Streamlit app.
- To preview a different stock, change `TICKER` in the second cell to any
  key in `STOCKS` and re-run.
